# Markowicz_rebalancing — analýza
Notebook nebol spustený. Dáta ani výsledky neboli vytvorené. Spustenie workflow vyžaduje samostatný súhlas.
Vzorka je **illustrative survivors-only**. Experiment mení súčasne odhad váh aj frekvenciu obchodovania, nie iba rebalansovanie k fixným váham.

In [ ]:
from pathlib import Path
import json
import hashlib
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT = Path.cwd()
assert (ROOT / 'config.yaml').is_file(), 'Otvorte notebook z koreňa repozitára'
RESULTS = ROOT / 'results'


## Voliteľné budúce spustenie
Po osobitnom povolení možno použiť `python run_analysis.py` alebo zmeniť prepínač v nasledujúcej bunke. Predvolene notebook nič nesťahuje. Pre citlivosť zvoľte oddelený adresár výsledkov. Existujúce výsledky sa neprepisujú.

In [ ]:
RUN_DOWNLOAD_AND_BACKTEST = False
if RUN_DOWNLOAD_AND_BACKTEST:
    from run_analysis import run
    RESULTS = run(ROOT / 'config.yaml', output=RESULTS)


In [ ]:
manifest_path = RESULTS / 'run_manifest.json'
if not manifest_path.is_file():
    raise RuntimeError('Výsledky ešte neexistujú; najprv osobitne povoľte a vykonajte beh.')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert manifest['status'] == 'completed', 'Neúplný alebo zlyhaný beh'
for artifact in manifest['artifacts']:
    path = RESULTS / artifact['path']
    assert hashlib.sha256(path.read_bytes()).hexdigest() == artifact['sha256'], f'Zmenený výstup: {path}'
display(Markdown(f"Beh: `{manifest['run_id']}`; citlivosť: `{manifest['sensitivity']}`"))


In [ ]:
metrics = pd.read_csv(RESULTS / 'portfolio_metrics.csv')
subperiods = pd.read_csv(RESULTS / 'subperiod_metrics.csv')
display(metrics)
display(subperiods)
display(pd.read_csv(RESULTS / 'data_quality_report.csv'))


Výnosy a drawdown za prvé obdobie vychádzajú z externých 10 000 USD vrátane úvodného poplatku. Neskoršie podperiódy začínajú close 2015-12-31 a 2020-12-31. Volatilita nepoužíva fiktívny denný výnos za úvodný poplatok. CSV výnosy sú desatinné zlomky; drawdown v tabuľke je kladná veľkosť straty.

In [ ]:
for name in ['portfolio_growth', 'risk_return', 'drawdowns', 'weight_evolution']:
    display(Image(filename=str(RESULTS / 'graphs' / f'{name}.png')))


In [ ]:
display(Markdown((RESULTS / 'conclusions.md').read_text(encoding='utf-8')))


## Interpretácia
Porovnajte osobitne konečnú hodnotu, Sharpe, drawdown, náklady, obrat a koncentráciu; nespoliehajte sa na jediného víťaza. Jedna historická cesta nepreukazuje štatistickú významnosť. Survivorship, vylúčenie neskorších IPO, dátové revízie, nestabilné odhady a chýbajúce náklady či dane obmedzujú závery. Benchmark nie je zahrnutý. Testy a validácia sa povoľujú samostatne.